Vitessce is a web-based visualization framework for exploring spatial omics data through linked views such as spatial plots, feature lists, cell set panels, and heatmaps. The [`vitessce-python`](https://github.com/vitessce/vitessce-python) library provides a Python interface for generating Vitessce configurations directly from analysis workflows and notebooks.

You can find related `vitessce-python` examples in the official notebook gallery, including [SpatialData Mouse Liver](https://python-docs.vitessce.io/notebooks/spatial_data_mouseliver.html) and [SpatialData Blobs](https://python-docs.vitessce.io/notebooks/spatial_data_blobs.html). SpatialData support in `vitessce-python` is already useful, but it still has some rough edges; see e.g. [`vitessce-python` issue #494](https://github.com/vitessce/vitessce-python/issues/494). Vitessce also does not yet support Zarr v3, which is why this notebook uses a Zarr v2-compatible workflow.

Create the environment for this notebook with:

```bash
cd vitessce
uv sync --python 3.12 --locked
source .venv/bin/activate
```

In [8]:
import harpy as hp
from spatialdata import read_zarr

sdata = hp.datasets.resolve_example()

sdata.write(
    "/Users/arne.defauw/VIB/DATA/vitessce_data/sdata_resolve.zarr", overwrite=True
)

sdata = read_zarr(sdata.path)

spatialdata_filepath = sdata.path

/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy_vitessce/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy_vitessce/lib/python3.12/site-packages/dask/utils.py:772: UserWarning: Could not serialize pd.DataFrame.attrs: Object of type Identity is not JSON serializable, defaulting to empty attributes.
  return meth(arg, *args, **kwargs)


INFO     The SpatialData object is not self-contained (i.e. it contains some elements that are Dask-backed from    
         locations outside /Users/arne.defauw/VIB/DATA/vitessce_data/sdata_resolve.zarr). Please see the           
         documentation of `is_self_contained()` to understand the implications of working with SpatialData objects 
         that are not self-contained.                                                                              
INFO     The Zarr backing store has been changed from None the new file path:                                      
         /Users/arne.defauw/VIB/DATA/vitessce_data/sdata_resolve.zarr                                              


version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy_vitessce/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


In [9]:
sdata["table_transcriptomics_cluster"].obs.head()

,cell_ID,fov_labels,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_2_genes,pct_counts_in_top_5_genes,n_counts,shapeSize,leiden
cells,,,,,,,,,,,
1_segmentation_mask_cc596c3c,1,segmentation_mask,14,2.708050,38,3.663562,39.473684,65.789474,38,1063,4
2_segmentation_mask_cc596c3c,2,segmentation_mask,11,2.484907,134,4.905275,76.865672,91.791045,134,2317,5
4_segmentation_mask_cc596c3c,4,segmentation_mask,8,2.197225,136,4.919981,66.911765,88.970588,136,2192,1
5_segmentation_mask_cc596c3c,5,segmentation_mask,10,2.397895,102,4.634729,79.411765,92.156863,102,1777,0
7_segmentation_mask_cc596c3c,7,segmentation_mask,24,3.218876,53,3.988984,26.415094,47.169811,53,1417,4


In [10]:
from vitessce import (
    CoordinationLevel as CL,
)
from vitessce import (
    SpatialDataWrapper,
    VitessceConfig,
    get_initial_coordination_scope_prefix,
)

# Create a VitessceConfig instance.
vc = VitessceConfig(schema_version="1.0.18", name="SpatialData Demo")

# Instantiate the wrapper class, specifying data fields of interest.
wrapper = SpatialDataWrapper(
    sdata_path=spatialdata_filepath,
    # The following paths are relative to the root of the SpatialData Zarr store on-disk.
    table_path="tables/table_transcriptomics_cluster",
    image_path="images/raw_image",
    obs_segmentations_path="labels/segmentation_mask",
    obs_feature_matrix_path="tables/table_transcriptomics_cluster/X",
    obs_set_paths=["tables/table_transcriptomics_cluster/obs/leiden"],
    obs_set_names=["Leiden"],
    region="segmentation_mask",
    coordinate_system="global",
    coordination_values={"obsType": "cell"},
)
# Add a new dataset to the Vitessce configuration,
# then add the wrapper class instance to this dataset.
dataset = vc.add_dataset(name="Mouse Liver").add_object(wrapper)

# Add views (visualizations) to the configuration.
spatial = vc.add_view("spatialBeta", dataset=dataset)
feature_list = vc.add_view("featureList", dataset=dataset)
layer_controller = vc.add_view("layerControllerBeta", dataset=dataset)
obs_sets = vc.add_view("obsSets", dataset=dataset)
heatmap = vc.add_view("heatmap", dataset=dataset)

[obs_color_encoding_scope] = vc.add_coordination("obsColorEncoding")
obs_color_encoding_scope.set_value("cellSetSelection")

initial_spatial_target_x = 1070
initial_spatial_target_y = 2144
initial_spatial_zoom = -3

vc.link_views_by_dict(
    [spatial, layer_controller],
    {
        "imageLayer": CL(
            [
                {
                    "photometricInterpretation": "BlackIsZero",
                    "imageChannel": CL(
                        [
                            {
                                "spatialTargetC": 0,
                                "spatialChannelColor": [255, 255, 255],
                                "spatialChannelWindow": [0, 4000],
                            }
                        ]
                    ),
                }
            ]
        ),
    },
    scope_prefix=get_initial_coordination_scope_prefix("A", "image"),
)

vc.link_views_by_dict(
    [spatial, layer_controller],
    {
        "segmentationLayer": CL(
            [
                {
                    "segmentationChannel": CL(
                        [
                            {
                                "obsColorEncoding": obs_color_encoding_scope,
                            }
                        ]
                    ),
                }
            ]
        ),
    },
    scope_prefix=get_initial_coordination_scope_prefix("A", "obsSegmentations"),
)

vc.link_views_by_dict(
    [spatial],
    {
        "spatialTargetX": initial_spatial_target_x,
        "spatialTargetY": initial_spatial_target_y,
        "spatialZoom": initial_spatial_zoom,
    },
    meta=False,
)

vc.link_views(
    [spatial, layer_controller, feature_list, obs_sets, heatmap],
    ["obsType"],
    ["cell"],
)
vc.link_views_by_dict(
    [feature_list, obs_sets, heatmap],
    {
        "obsColorEncoding": obs_color_encoding_scope,
    },
    meta=False,
)

# Layout the views in a grid arrangement.
vc.layout((spatial / heatmap) | (layer_controller / (feature_list | obs_sets)))

In [11]:
from IPython.display import HTML, display

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))